# 04 — Post-fire recovery
## Galičica after the August 2024 wildfire

In this practical we move from **mapping the disturbance** to asking a management question:

> **How much spectral recovery is visible one year after the 2024 fire, and does recovery differ between more- and less-strongly affected areas?**

We will:
- reuse the canonical Galičica AOI and EFFIS reference polygon;
- derive fixed 2024 dNBR-based impact zones;
- compare **NBR, NDVI and NDMI** through time;
- use a stable late-summer seasonal window every year;
- estimate a simple **recovery fraction** relative to the pre-fire baseline;
- compare burned areas with a nearby reference zone.

> **Important:** spectral recovery is not the same as ecological recovery. Greening can return before vegetation structure, habitat quality, biomass, species composition or fuel structure recover.

## 1. Imports and Earth Engine

In [ ]:
from pathlib import Path
import ee
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import folium

GEE_PROJECT_ID = "ee-andreydara"

try:
    ee.Initialize(project=GEE_PROJECT_ID)
except Exception:
    ee.Authenticate()
    ee.Initialize(project=GEE_PROJECT_ID)

print("Earth Engine ready.")

## 2. Load the canonical AOI and EFFIS reference polygon

Both files are committed directly to the course repository:

- `data/aoi/galicica_aoi.geojson`
- `data/effis/Galicica.gpkg`

No external download is required.

In [ ]:
repo_root = Path.home() / "mystorage" / "fire-school"

def find_course_file(relative_path):
    candidates = [
        repo_root / relative_path,
        Path.cwd() / relative_path,
        Path.cwd().parent / relative_path,
    ]
    path = next((p for p in candidates if p.exists()), None)
    if path is None:
        raise FileNotFoundError(
            f"{relative_path} was not found. Run git pull in the fire-school repository."
        )
    return path

AOI_PATH = find_course_file(Path("data/aoi/galicica_aoi.geojson"))
EFFIS_PATH = find_course_file(Path("data/effis/Galicica.gpkg"))

aoi_gdf = gpd.read_file(AOI_PATH).to_crs("EPSG:4326")
effis = gpd.read_file(EFFIS_PATH).to_crs("EPSG:4326").copy()

aoi_geom = aoi_gdf.geometry.iloc[0]
AOI = ee.Geometry(aoi_geom.__geo_interface__)

effis["FIREDATE"] = pd.to_datetime(effis["FIREDATE"], errors="coerce")
effis["AREA_HA"] = pd.to_numeric(effis["AREA_HA"], errors="coerce")

target = effis[effis["id"].astype(str) == "240575"].copy()
if target.empty:
    raise RuntimeError("EFFIS reference polygon 240575 was not found.")

target_geom = target.geometry.iloc[0]
TARGET = ee.Geometry(target_geom.__geo_interface__)

print("AOI area (km²):", round(AOI.area().divide(1e6).getInfo(), 1))
print("EFFIS target area (ha):", float(target.iloc[0]["AREA_HA"]))

## 3. Build comparable Sentinel-2 seasonal composites

For recovery analysis, consistency matters. We therefore use the same seasonal window each year:

**20 August – 15 October**

We keep only the bands needed below before compositing. This also avoids Sentinel-2 auxiliary-band schema differences across processing baselines.

In [ ]:
SEASON_START = (8, 20)
SEASON_END = (10, 16)
MAX_CLOUD = 60

def mask_s2_scl(img):
    scl = img.select("SCL")
    bad = (
        scl.eq(3)      # cloud shadow
        .Or(scl.eq(8)) # cloud medium probability
        .Or(scl.eq(9)) # cloud high probability
        .Or(scl.eq(10))# cirrus
        .Or(scl.eq(11))# snow / ice
    )
    return (
        img.updateMask(bad.Not())
        .select(["B4", "B8", "B11", "B12"])
    )

def seasonal_composite(year):
    start = f"{year}-{SEASON_START[0]:02d}-{SEASON_START[1]:02d}"
    end = f"{year}-{SEASON_END[0]:02d}-{SEASON_END[1]:02d}"

    col = (
        ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
        .filterBounds(AOI)
        .filterDate(start, end)
        .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", MAX_CLOUD))
        .map(mask_s2_scl)
    )

    return col.median().clip(AOI), col.size()

def add_indices(img):
    ndvi = img.normalizedDifference(["B8", "B4"]).rename("NDVI")
    nbr = img.normalizedDifference(["B8", "B12"]).rename("NBR")
    ndmi = img.normalizedDifference(["B8", "B11"]).rename("NDMI")
    return ee.Image.cat([nbr, ndvi, ndmi])

img_2023, n_2023 = seasonal_composite(2023)
img_2024, n_2024 = seasonal_composite(2024)

print("2023 scenes:", n_2023.getInfo())
print("2024 scenes:", n_2024.getInfo())

## 4. Define fixed impact zones from the 2024 disturbance

We use **2023 late summer** as the pre-fire comparison and **2024 late summer** as the post-fire period.

The zones are deliberately simple:

- **strong impact:** dNBR ≥ 0.44 inside the EFFIS polygon;
- **lower/moderate impact:** 0.10 ≤ dNBR < 0.44 inside the EFFIS polygon;
- **reference:** nearby vegetated pixels outside the EFFIS polygon that show little 2023→2024 NBR change.

The same masks are then held fixed for every year of the recovery time series.

> The reference zone is a teaching control, not a perfect matched ecological control.

In [ ]:
nbr_2023 = img_2023.normalizedDifference(["B8", "B12"]).rename("NBR_2023")
nbr_2024 = img_2024.normalizedDifference(["B8", "B12"]).rename("NBR_2024")
dnbr_2024 = nbr_2023.subtract(nbr_2024).rename("dNBR_2024")

target_mask = ee.Image.constant(1).clip(TARGET).selfMask()

strong_mask = (
    target_mask
    .updateMask(dnbr_2024.gte(0.44))
    .rename("strong")
)

moderate_mask = (
    target_mask
    .updateMask(dnbr_2024.gte(0.10).And(dnbr_2024.lt(0.44)))
    .rename("moderate")
)

# Nearby ring around the fire, excluding the fire polygon itself.
reference_geom = TARGET.buffer(5000).difference(TARGET.buffer(1000)).intersection(AOI)

worldcover = ee.ImageCollection("ESA/WorldCover/v200").first().select("Map")

# Tree cover, shrubland and grassland only.
vegetated = (
    worldcover.eq(10)
    .Or(worldcover.eq(20))
    .Or(worldcover.eq(30))
)

reference_mask = (
    ee.Image.constant(1)
    .clip(reference_geom)
    .selfMask()
    .updateMask(vegetated)
    .updateMask(dnbr_2024.abs().lt(0.10))
    .rename("reference")
)

print("Impact masks ready.")

## 5. Inspect the zones

In [ ]:
def add_ee_layer(m, ee_image, vis_params, name):
    map_id = ee_image.getMapId(vis_params)
    folium.raster_layers.TileLayer(
        tiles=map_id["tile_fetcher"].url_format,
        attr="Google Earth Engine",
        name=name,
        overlay=True,
        control=True,
    ).add_to(m)

zone_image = (
    ee.Image(0)
    .where(reference_mask, 1)
    .where(moderate_mask, 2)
    .where(strong_mask, 3)
    .selfMask()
)

centroid = aoi_geom.centroid
CENTER = [centroid.y, centroid.x]

m = folium.Map(location=CENTER, zoom_start=10, tiles="CartoDB positron")

folium.GeoJson(
    aoi_gdf,
    name="Course AOI",
    style_function=lambda _: {
        "color": "black",
        "weight": 2,
        "fillOpacity": 0.0,
    },
).add_to(m)

target_map = target.copy()
target_map["FIREDATE"] = target_map["FIREDATE"].dt.strftime("%Y-%m-%d")

folium.GeoJson(
    target_map,
    name="EFFIS 240575",
    style_function=lambda _: {
        "color": "#333333",
        "weight": 2,
        "fillOpacity": 0.0,
    },
).add_to(m)

add_ee_layer(
    m,
    zone_image,
    {
        "min": 1,
        "max": 3,
        "palette": ["2c7bb6", "fdae61", "d7191c"],
    },
    "Reference / moderate / strong",
)

folium.LayerControl().add_to(m)
m

### Check the logic

Before continuing:

1. Are the strong-impact pixels spatially coherent?
2. Does the reference area look plausibly comparable?
3. What biases could arise from using one nearby ring as a control?
4. Why do we keep the impact masks fixed through time?

## 6. Calculate zone areas

In [ ]:
def mask_area_km2(mask, geometry=AOI):
    return (
        ee.Image.pixelArea()
        .updateMask(mask)
        .reduceRegion(
            reducer=ee.Reducer.sum(),
            geometry=geometry,
            scale=20,
            maxPixels=1e9,
            bestEffort=True,
        )
        .get("area")
    )

zone_areas = pd.DataFrame({
    "zone": ["Reference", "Moderate/lower impact", "Strong impact"],
    "area_km2": [
        ee.Number(mask_area_km2(reference_mask)).divide(1e6).getInfo(),
        ee.Number(mask_area_km2(moderate_mask)).divide(1e6).getInfo(),
        ee.Number(mask_area_km2(strong_mask)).divide(1e6).getInfo(),
    ],
})

zone_areas["area_km2"] = zone_areas["area_km2"].round(2)
zone_areas

## 7. Build the annual recovery time series

We calculate mean **NBR**, **NDVI** and **NDMI** for each fixed zone from **2019–2025**.

Why all three?

- **NBR:** especially sensitive to burn disturbance and post-fire structure/moisture change;
- **NDVI:** greenness / photosynthetic vegetation response;
- **NDMI:** canopy/vegetation moisture-related response.

Their trajectories need not recover at the same rate.

### A small Earth Engine implementation detail

Instead of launching a separate `reduceRegion()` for every zone, index and year, we stack all zone/index bands and use **one reduction per year**, evaluated sequentially. This avoids Earth Engine's concurrent-aggregation limit and is also a cleaner pattern for larger analyses.

In [ ]:
ZONES = {
    "Reference": reference_mask,
    "Moderate/lower impact": moderate_mask,
    "Strong impact": strong_mask,
}

def safe_name(zone_name):
    return (
        zone_name.lower()
        .replace("/", "_")
        .replace(" ", "_")
    )

def annual_stats(year):
    """
    Calculate all 9 zone/index means with ONE reduceRegion call.

    This is intentionally evaluated year-by-year on the client side.
    Building all zone reductions for all years into one FeatureCollection
    can trigger Earth Engine's 'Too many concurrent aggregations' limit.
    """
    img, n_scenes = seasonal_composite(year)
    indices = add_indices(img)

    zone_stacks = []

    for zone_name, mask in ZONES.items():
        safe = safe_name(zone_name)

        zone_stack = (
            indices
            .updateMask(mask)
            .rename([
                f"{safe}_NBR",
                f"{safe}_NDVI",
                f"{safe}_NDMI",
            ])
        )

        zone_stacks.append(zone_stack)

    all_zone_indices = ee.Image.cat(zone_stacks)

    stats = all_zone_indices.reduceRegion(
        reducer=ee.Reducer.mean(),
        geometry=AOI,
        scale=20,
        maxPixels=1e9,
        bestEffort=True,
        tileScale=2,
    )

    # Evaluate one year at a time to avoid concurrent aggregation pressure.
    result = ee.Dictionary(stats).set({
        "year": year,
        "n_scenes": n_scenes,
    }).getInfo()

    return result

years = list(range(2019, 2026))

records = []
for year in years:
    print("Processing", year, "...")
    records.append(annual_stats(year))

ts = pd.DataFrame(records).sort_values("year")
ts

## 8. Plot NBR recovery

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

for zone in ZONES:
    safe = zone.lower().replace("/", "_").replace(" ", "_")
    ax.plot(
        ts["year"],
        ts[f"{safe}_NBR"],
        marker="o",
        label=zone,
    )

ax.axvline(2024, linestyle="--", alpha=0.7)
ax.set_xlabel("Year")
ax.set_ylabel("Mean late-summer NBR")
ax.set_title("Post-fire spectral trajectory — NBR")
ax.legend()
ax.grid(alpha=0.25)
plt.show()

## 9. Compare NDVI and NDMI

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

for zone in ["Moderate/lower impact", "Strong impact"]:
    safe = zone.lower().replace("/", "_").replace(" ", "_")
    ax.plot(
        ts["year"],
        ts[f"{safe}_NDVI"],
        marker="o",
        label=f"{zone} — NDVI",
    )

ax.axvline(2024, linestyle="--", alpha=0.7)
ax.set_xlabel("Year")
ax.set_ylabel("Mean late-summer NDVI")
ax.set_title("Post-fire spectral trajectory — NDVI")
ax.legend()
ax.grid(alpha=0.25)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

for zone in ["Moderate/lower impact", "Strong impact"]:
    safe = zone.lower().replace("/", "_").replace(" ", "_")
    ax.plot(
        ts["year"],
        ts[f"{safe}_NDMI"],
        marker="o",
        label=f"{zone} — NDMI",
    )

ax.axvline(2024, linestyle="--", alpha=0.7)
ax.set_xlabel("Year")
ax.set_ylabel("Mean late-summer NDMI")
ax.set_title("Post-fire spectral trajectory — NDMI")
ax.legend()
ax.grid(alpha=0.25)
plt.show()

## 10. Estimate a simple one-year recovery fraction

For each burned zone we define:

[
\text{Recovery fraction} =
\frac{I_{2025} - I_{2024}}
{I_{baseline} - I_{2024}}
\times 100
]

where the baseline is the mean of **2019–2023**.

Interpretation:

- **0%**: no return toward the pre-fire baseline;
- **100%**: back to the pre-fire mean;
- **>100%**: exceeds the pre-fire mean;
- **negative**: moved farther away from the baseline.

This is a simple spectral metric, not an ecological recovery score.

In [ ]:
def recovery_table(index_name):
    rows = []

    for zone in ["Moderate/lower impact", "Strong impact"]:
        safe = zone.lower().replace("/", "_").replace(" ", "_")
        col = f"{safe}_{index_name}"

        baseline = ts.loc[ts["year"].between(2019, 2023), col].mean()
        fire = ts.loc[ts["year"] == 2024, col].iloc[0]
        y2025 = ts.loc[ts["year"] == 2025, col].iloc[0]

        disturbance = baseline - fire
        recovery = y2025 - fire

        recovery_pct = (
            recovery / disturbance * 100
            if disturbance != 0
            else float("nan")
        )

        rows.append({
            "zone": zone,
            "index": index_name,
            "baseline_2019_2023": baseline,
            "2024": fire,
            "2025": y2025,
            "disturbance_from_baseline": disturbance,
            "recovery_2025": recovery,
            "recovery_fraction_pct": recovery_pct,
        })

    return pd.DataFrame(rows)

recovery = pd.concat(
    [recovery_table("NBR"), recovery_table("NDVI"), recovery_table("NDMI")],
    ignore_index=True,
)

numeric_cols = recovery.select_dtypes("number").columns
recovery[numeric_cols] = recovery[numeric_cols].round(3)

recovery

## 11. Interpretation exercise — 15 minutes

In pairs, answer:

1. Which zone experienced the larger spectral disturbance in 2024?
2. Which index shows the strongest apparent one-year recovery?
3. Does the reference zone remain relatively stable?
4. Do NBR, NDVI and NDMI tell exactly the same story?
5. What could cause apparent recovery without true ecosystem recovery?
6. What field measurements would you want before claiming successful ecological recovery?
7. For park management, what would make you revisit the site in 2026?

## 12. Stretch tasks

Choose one:

### A — Recovery map
Calculate `NBR_2025 - NBR_2024` and map where recovery is fastest and slowest.

### B — Relative dNBR (RdNBR)
Investigate whether normalizing disturbance by pre-fire vegetation condition changes the severity pattern.

### C — Different baseline
Use only 2022–2023 as the baseline. How much does the recovery fraction change?

### D — Land-cover stratification
Compare recovery inside tree cover versus shrub/grassland.

### E — Longer time horizon
When later years become available, extend the same fixed seasonal analysis beyond 2025.

## 13. Output for the Galičica capstone

Keep:
- the impact-zone map;
- the NBR recovery trajectory;
- one NDVI or NDMI comparison figure;
- the recovery-fraction table;
- **three defensible findings**;
- **one limitation**;
- **one management-relevant interpretation**.

The next practical moves from **what burned and how it recovered** to **weather, vegetation condition and fuel-related EO proxies before fire**.